In [4]:
import pandas as pd

In [5]:
# load the benchmark results from https://github.com/ulysseLeclanche/EDAM_ground_truth/raw/refs/heads/main/EDAM_terms_URI_ground_truth_validated.tsv 

df = pd.read_csv("https://github.com/ulysseLeclanche/EDAM_ground_truth/raw/refs/heads/main/EDAM_terms_URI_ground_truth_validated.tsv", sep="\t")


In [6]:
df

,EDAM term URI,candidate EDAM term label,tools,annotation,domain,edam_category
0,http://edamontology.org/data_3707,Biodiversity data,MetaBAT2,abundance profiling,Metagenomic,data
1,http://edamontology.org/data_3707,Biodiversity data,"Kraken2, mOTUs",abundance table,Metagenomic,data
2,http://edamontology.org/data_0925,Sequence Assembly,"Anvi’o, CONCOCT, MEGAHIT, MetaBAT2, checkM2",contigs,Metagenomic,data
3,http://edamontology.org/operation_3672,Gene functional annotation,Anvi’o,functional annotations,Metagenomic,data
4,http://edamontology.org/data_2968,Image,Anvi’o,image,Metagenomic,data
...,...,...,...,...,...,...
399,http://edamontology.org/operation_3443,Image analysis,CellPose,segmentation 2d,Microscopy,operations
400,http://edamontology.org/operation_3443,Image analysis,CellPose,segmentation 3d,Microscopy,operations
401,http://edamontology.org/topic_2229,Cell biology,CellPose,cell segmentation,Microscopy,topics
402,http://edamontology.org/topic_3382,Imaging,EcClem,correlative microscopy,Microscopy,topics


In [54]:
# list the unique tools in the benchmark
tools = set()
for tool in df["tools"].unique():
    for t in tool.split(","):
        tools.add(t.strip())

print("Unique tools in the benchmark ({}):".format(len(tools)))
for t in sorted(tools):
    print("-", t)

Unique tools in the benchmark (44):
- AFNI
- Anvi’o
- Bedtools_annotate
- Big-FISH
- COBRApy
- CONCOCT
- Canu
- CarveMe
- CellPose
- ClusterProfiler
- Cytoscape
- EcClem
- FMRIprep
- FSL
- FastME
- Fluxer
- Freesurfer
- GSEA
- GTDB-Tk
- Kraken2
- MAFFT
- MEGAHIT
- MOFA2
- MULTIXRANK
- MetaBAT2
- Minimap2
- Monocle3
- PhyML
- RWR
- SNF
- SPM
- STAR
- Samtools_index
- Scanpy
- Seurat
- Sniffles2
- String.db
- bwa
- checkM2
- deconvolutionLab2
- deseq2
- eggNOG-mapper
- htseq-count
- mOTUs


In [55]:
# generate a new dataframe with one row per tool and the corresponding EDAM terms, grouped by category (operations, topics, data, formats), with columns: tool, EDAM operation URI, EDAM topic URI, EDAM data URI, EDAM format URI
def generate_tool_term_mapping(df):
    mapping = []
    for _, row in df.iterrows():
        tool_list = [t.strip() for t in row["tools"].split(",")]
        for tool in tool_list:
            cat = row["edam_category"]
            mapping.append({
                "tool": tool,
                f"EDAM {cat} URI": row["EDAM term URI"]
            })

tool_term_mapping = generate_tool_term_mapping(df)
print(tool_term_mapping.head())

# group by tool and aggregate the EDAM URIs into lists per category
tool_term_mapping_grouped = tool_term_mapping.groupby("tool").agg({
    "EDAM operations URI": lambda x: list(x.dropna()),
    "EDAM topics URI": lambda x: list(x.dropna()),
    "EDAM data URI": lambda x: list(x.dropna()),
    "EDAM formats URI": lambda x: list(x.dropna())
}).reset_index()

tool_term_mapping_grouped
tool_term_mapping_grouped.to_csv("tool_term_mapping_grouped.csv", index=False)

AttributeError: 'NoneType' object has no attribute 'head'

In [ ]:
def compute_perf_metrics(json_data, gt_df):
    """
    For each tool and each annotation type for LLM files, calculate TP, FP, FN, 
    TN, precision, recall, F1_score, Total annotation retained in consensus 
    expert-LLM, not retained and mixed annotations count.  
    """

    results = []

    for entry in json_data:
        print(f"Processing tool: {entry['tool_name']}")  # Debug statement

        # Ensure tool name is consistent with GT
        tool = entry["tool_name"].strip()
        if tool not in gt_df["tool"].values:
            print(f"Warning: Tool '{tool}' not found in GT dataframe. Skipping.")
            continue 

        ops = gt_df[gt_df["tool"] == tool]["EDAM operations URI"].values[0]
        topics = gt_df[gt_df["tool"] == tool]["EDAM topics URI"].values[0]
        data = gt_df[gt_df["tool"] == tool]["EDAM data URI"].values[0]
        formats = gt_df[gt_df["tool"] == tool]["EDAM formats URI"].values[0]

        gt_terms = set(ops + topics + data + formats)
        print(gt_terms)

        entry_terms = set(entry["all_edam"]  )
        print(entry_terms)

        TP_set = entry_terms & gt_terms # predicted and in GT
        #print(f"TP_set (predicted and in GT): {TP_set}")
        FP_set = entry_terms - gt_terms # predicted but not in GT
        #print(f"FP_set (predicted but not in GT): {FP_set}")
        TN_set = set() # not predicted and not in GT (not computable without a full list of possible terms)
        FN_set = gt_terms - entry_terms # not predicted but in GT
        #print(f"FN_set (not predicted but in GT): {FN_set}")

        TP = len(TP_set)
        FP = len(FP_set)
        FN = len(FN_set)
        TN = 0

        precision = round(TP / (TP + FP) if (TP + FP) else 0, 2)
        recall = round(TP / (TP + FN) if (TP + FN) else 0, 2)
        f1_score = round(
            (2 * precision * recall / (precision + recall)
                if (precision + recall) else 0),
            2,
        )

        results.append({
            "tool_name": tool,

            # Confusion metrics
            "TP": TP,
            "FP": FP,
            "FN": FN,
            "TN": TN,
            "precision": precision,
            "recall": recall,
            "f1_score": f1_score,

            # Detailed annotation lists
            "TP_annotations": sorted(TP_set),
            "FP_annotations": sorted(FP_set),
            "FN_annotations": sorted(FN_set),
        })

    return results

In [ ]:
v1 = [{
    "tool_name": "Kraken2",
    "all_edam" : ["http://edamontology.org/topic_3837", 
                "http://edamontology.org/topic_3174",
                "http://edamontology.org/operation_0547",
                "http://edamontology.org/operation_0327", 
                "http://edamontology.org/topic_0084", 
                "http://edamontology.org/operation_3460", 
                "http://edamontology.org/operation_0538", 
                "http://edamontology.org/topic_3943", 
                "http://edamontology.org/data_1872", 
                "http://edamontology.org/operation_0542"]
}]
res = compute_perf_metrics(v1, tool_term_mapping_grouped)

Processing tool: Kraken2
{'http://edamontology.org/data_3707', 'http://edamontology.org/topic_3168', 'http://edamontology.org/format_1930', 'http://edamontology.org/format_3475', 'http://edamontology.org/operation_3460', 'http://edamontology.org/topic_3174', 'http://edamontology.org/topic_0080', 'http://edamontology.org/data_1872', 'http://edamontology.org/format_2331', 'http://edamontology.org/operation_2403'}
{'http://edamontology.org/topic_3837', 'http://edamontology.org/operation_0327', 'http://edamontology.org/topic_0084', 'http://edamontology.org/topic_3943', 'http://edamontology.org/operation_0538', 'http://edamontology.org/operation_0547', 'http://edamontology.org/operation_3460', 'http://edamontology.org/data_1872', 'http://edamontology.org/operation_0542', 'http://edamontology.org/topic_3174'}
TP_set (predicted and in GT): {'http://edamontology.org/topic_3174', 'http://edamontology.org/operation_3460', 'http://edamontology.org/data_1872'}
FP_set (predicted but not in GT): {'h

In [60]:
for r in res:
    print(f"Tool: {r['tool_name']}")
    print(f"  TP: {r['TP']}, FP: {r['FP']}, FN: {r['FN']}, TN: {r['TN']}")
    print(f"  Precision: {r['precision']}, Recall: {r['recall']}, F1 Score: {r['f1_score']}")
    print(f"  TP Annotations: {r['TP_annotations']}")
    print(f"  FP Annotations: {r['FP_annotations']}")
    print(f"  FN Annotations: {r['FN_annotations']}")

Tool: Kraken2
  TP: 3, FP: 7, FN: 7, TN: 0
  Precision: 0.3, Recall: 0.3, F1 Score: 0.3
  TP Annotations: ['http://edamontology.org/data_1872', 'http://edamontology.org/operation_3460', 'http://edamontology.org/topic_3174']
  FP Annotations: ['http://edamontology.org/operation_0327', 'http://edamontology.org/operation_0538', 'http://edamontology.org/operation_0542', 'http://edamontology.org/operation_0547', 'http://edamontology.org/topic_0084', 'http://edamontology.org/topic_3837', 'http://edamontology.org/topic_3943']
  FN Annotations: ['http://edamontology.org/data_3707', 'http://edamontology.org/format_1930', 'http://edamontology.org/format_2331', 'http://edamontology.org/format_3475', 'http://edamontology.org/operation_2403', 'http://edamontology.org/topic_0080', 'http://edamontology.org/topic_3168']


In [61]:
v2 = [{
    "tool_name": "Kraken2",
    "all_edam" : ["http://edamontology.org/topic_3174", 
                "http://edamontology.org/topic_3837",
                "http://edamontology.org/topic_3941",
                "http://edamontology.org/topic_3697",
                "http://edamontology.org/topic_0194", 
                "http://edamontology.org/operation_3460",
                "http://edamontology.org/operation_3219",
                "http://edamontology.org/operation_0499",
                "http://edamontology.org/operation_0547",
                "http://edamontology.org/operation_3947", 
                "http://edamontology.org/data_0850",
                "http://edamontology.org/data_0867",
                "http://edamontology.org/data_1872",
                "http://edamontology.org/data_3707",
                "http://edamontology.org/data_0963",
                "http://edamontology.org/format_3772",
                "http://edamontology.org/format_1929",
                "http://edamontology.org/format_2054",
                "http://edamontology.org/format_1332",
                "http://edamontology.org/format_2036"
]
}]
res = compute_perf_metrics(v2, tool_term_mapping_grouped)

Processing tool: Kraken2
{'http://edamontology.org/data_3707', 'http://edamontology.org/topic_3168', 'http://edamontology.org/format_1930', 'http://edamontology.org/format_3475', 'http://edamontology.org/operation_3460', 'http://edamontology.org/topic_3174', 'http://edamontology.org/topic_0080', 'http://edamontology.org/data_1872', 'http://edamontology.org/format_2331', 'http://edamontology.org/operation_2403'}
{'http://edamontology.org/format_2054', 'http://edamontology.org/format_3772', 'http://edamontology.org/operation_3947', 'http://edamontology.org/topic_3174', 'http://edamontology.org/topic_3837', 'http://edamontology.org/format_2036', 'http://edamontology.org/operation_0499', 'http://edamontology.org/topic_3697', 'http://edamontology.org/format_1929', 'http://edamontology.org/topic_0194', 'http://edamontology.org/format_1332', 'http://edamontology.org/topic_3941', 'http://edamontology.org/data_0850', 'http://edamontology.org/data_0867', 'http://edamontology.org/data_3707', 'htt

In [62]:
for r in res:
    print(f"Tool: {r['tool_name']}")
    print(f"  TP: {r['TP']}, FP: {r['FP']}, FN: {r['FN']}, TN: {r['TN']}")
    print(f"  Precision: {r['precision']}, Recall: {r['recall']}, F1 Score: {r['f1_score']}")
    print(f"  TP Annotations: {r['TP_annotations']}")
    print(f"  FP Annotations: {r['FP_annotations']}")
    print(f"  FN Annotations: {r['FN_annotations']}")

Tool: Kraken2
  TP: 4, FP: 16, FN: 6, TN: 0
  Precision: 0.2, Recall: 0.4, F1 Score: 0.27
  TP Annotations: ['http://edamontology.org/data_1872', 'http://edamontology.org/data_3707', 'http://edamontology.org/operation_3460', 'http://edamontology.org/topic_3174']
  FP Annotations: ['http://edamontology.org/data_0850', 'http://edamontology.org/data_0867', 'http://edamontology.org/data_0963', 'http://edamontology.org/format_1332', 'http://edamontology.org/format_1929', 'http://edamontology.org/format_2036', 'http://edamontology.org/format_2054', 'http://edamontology.org/format_3772', 'http://edamontology.org/operation_0499', 'http://edamontology.org/operation_0547', 'http://edamontology.org/operation_3219', 'http://edamontology.org/operation_3947', 'http://edamontology.org/topic_0194', 'http://edamontology.org/topic_3697', 'http://edamontology.org/topic_3837', 'http://edamontology.org/topic_3941']
  FN Annotations: ['http://edamontology.org/format_1930', 'http://edamontology.org/format_23